# 09. Kimi K3 — complete 93-layer text architecture at reduced tensor width

This notebook preserves the released Kimi-K3 text topology and reduces only tensor widths, batch size and sequence length.

Preserved structural constants:

- 93 decoder layers
- 96 attention heads
- exactly 69 KDA layers + 24 Gated-MLA layers at the released 1-based layer indices
- KDA short convolution kernel = 4 and gate lower bound = -5
- Block Attention Residuals with block size 12
- layer 1 dense SiTU-GLU; layers 2–93 Stable LatentMoE
- 896 routed experts, top-16 routing and 2 shared experts on every MoE layer
- sigmoid router, renormalized selected weights, SiTU beta values 4 and 25

Reduced: hidden/head/latent/intermediate widths, Q/KV low-rank widths, batch and sequence length.

In [ ]:
import math

import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(11)
device = torch.device("cpu")
torch.set_num_threads(min(2, torch.get_num_threads()))
print("device:", device)

In [ ]:
K3_FULL_ATTENTION_LAYERS = [
    4, 8, 12, 16, 20, 24, 28, 32,
    36, 40, 44, 48, 52, 56, 60, 64,
    68, 72, 76, 80, 84, 88, 92, 93,
]
K3_KDA_LAYERS = [
    layer_number
    for layer_number in range(1, 94)
    if layer_number not in K3_FULL_ATTENTION_LAYERS
]

assert len(K3_FULL_ATTENTION_LAYERS) == 24
assert len(K3_KDA_LAYERS) == 69
assert len(K3_FULL_ATTENTION_LAYERS) + len(K3_KDA_LAYERS) == 93

K3_DEPTH = 93
K3_HEADS = 96
K3_EXPERTS = 896
K3_TOP_K = 16
K3_SHARED_EXPERTS = 2
K3_ATTN_RES_BLOCK_SIZE = 12

print("KDA/MLA:", len(K3_KDA_LAYERS), len(K3_FULL_ATTENTION_LAYERS))

## 1. KDA recurrence

The matrix state is decayed channel-wise, the value predicted by the current key is subtracted from the target value, and the delta error is written back before reading with the query.

In [ ]:
class KimiRMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-5):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(dim))
        self.eps = eps

    def forward(self, x):
        x_float = x.float()
        scale = torch.rsqrt(
            x_float.square().mean(dim=-1, keepdim=True)
            + self.eps
        )
        return (x_float * scale).to(x.dtype) * self.weight


def kda_scan(q, k, v, alpha, beta):
    batch_size, heads, sequence_length, key_dim = q.shape
    value_dim = v.size(-1)

    state = torch.zeros(
        batch_size,
        heads,
        key_dim,
        value_dim,
        device=q.device,
        dtype=q.dtype,
    )
    outputs = []

    for token_index in range(sequence_length):
        q_t = F.normalize(q[:, :, token_index], dim=-1)
        k_t = F.normalize(k[:, :, token_index], dim=-1)
        v_t = v[:, :, token_index]

        decayed_state = (
            alpha[:, :, token_index, :, None]
            * state
        )
        predicted_value = torch.einsum(
            "bhkv,bhk->bhv",
            decayed_state,
            k_t,
        )
        delta_error = v_t - predicted_value

        state = (
            decayed_state
            + beta[:, :, token_index, None, None]
            * torch.einsum(
                "bhk,bhv->bhkv",
                k_t,
                delta_error,
            )
        )
        output_t = torch.einsum(
            "bhkv,bhk->bhv",
            state,
            q_t,
        )
        outputs.append(output_t)

    return torch.stack(outputs, dim=2)


class KimiKDA(nn.Module):
    def __init__(
        self,
        model_dim=8,
        heads=96,
        head_dim=2,
        decay_rank=2,
        short_kernel=4,
    ):
        super().__init__()

        assert heads == 96
        assert short_kernel == 4

        self.heads = heads
        self.head_dim = head_dim
        self.short_kernel = short_kernel
        self.gate_lower_bound = -5.0
        projection_width = heads * head_dim

        self.q_projection = nn.Linear(
            model_dim,
            projection_width,
            bias=False,
        )
        self.k_projection = nn.Linear(
            model_dim,
            projection_width,
            bias=False,
        )
        self.v_projection = nn.Linear(
            model_dim,
            projection_width,
            bias=False,
        )

        self.q_conv = nn.Conv1d(
            projection_width,
            projection_width,
            short_kernel,
            groups=projection_width,
        )
        self.k_conv = nn.Conv1d(
            projection_width,
            projection_width,
            short_kernel,
            groups=projection_width,
        )
        self.v_conv = nn.Conv1d(
            projection_width,
            projection_width,
            short_kernel,
            groups=projection_width,
        )

        self.f_a = nn.Linear(model_dim, decay_rank, bias=False)
        self.f_b = nn.Linear(
            decay_rank,
            projection_width,
            bias=True,
        )
        self.beta_projection = nn.Linear(
            model_dim,
            heads,
            bias=True,
        )
        self.log_head_scale = nn.Parameter(torch.zeros(heads))

        self.output_norm = KimiRMSNorm(head_dim)
        self.output_gate = nn.Linear(
            model_dim,
            projection_width,
            bias=True,
        )
        self.output_projection = nn.Linear(
            projection_width,
            model_dim,
            bias=False,
        )

    def _causal_short_conv(self, x, convolution):
        x = x.transpose(1, 2)
        x = F.pad(x, (self.short_kernel - 1, 0))
        x = convolution(x).transpose(1, 2)
        return F.silu(x)

    def _split_heads(self, x):
        batch_size, sequence_length, _ = x.shape
        return x.view(
            batch_size,
            sequence_length,
            self.heads,
            self.head_dim,
        ).transpose(1, 2)

    def forward(self, hidden):
        q = self._split_heads(
            self._causal_short_conv(
                self.q_projection(hidden),
                self.q_conv,
            )
        )
        k = self._split_heads(
            self._causal_short_conv(
                self.k_projection(hidden),
                self.k_conv,
            )
        )
        v = self._split_heads(
            self._causal_short_conv(
                self.v_projection(hidden),
                self.v_conv,
            )
        )

        decay_pre = self.f_b(F.silu(self.f_a(hidden)))
        decay_pre = self._split_heads(decay_pre)
        head_scale = torch.exp(self.log_head_scale)[
            None, :, None, None
        ]
        log_decay = self.gate_lower_bound * torch.sigmoid(
            head_scale * decay_pre
        )
        alpha = torch.exp(log_decay)
        beta = torch.sigmoid(
            self.beta_projection(hidden)
        ).transpose(1, 2)

        scanned = kda_scan(q, k, v, alpha, beta)
        scanned = self.output_norm(scanned)
        scanned = scanned.transpose(1, 2).contiguous().flatten(2)

        gate = torch.sigmoid(self.output_gate(hidden))
        return self.output_projection(gate * scanned)

## 2. Gated MLA with NoPE

The K3 global-attention path uses low-rank Q/KV projections, no positional encoding on this MLA path, and a full-rank output gate.

In [ ]:
class KimiGatedMLA(nn.Module):
    def __init__(
        self,
        model_dim=8,
        heads=96,
        head_dim=2,
        q_rank=4,
        kv_rank=4,
    ):
        super().__init__()

        assert heads == 96
        self.heads = heads
        self.head_dim = head_dim
        projection_width = heads * head_dim

        self.q_down = nn.Linear(model_dim, q_rank, bias=False)
        self.q_norm = KimiRMSNorm(q_rank)
        self.q_up = nn.Linear(q_rank, projection_width, bias=False)

        self.kv_down = nn.Linear(model_dim, kv_rank, bias=False)
        self.kv_norm = KimiRMSNorm(kv_rank)
        self.kv_up = nn.Linear(
            kv_rank,
            2 * projection_width,
            bias=False,
        )

        self.output_gate = nn.Linear(
            model_dim,
            projection_width,
            bias=True,
        )
        self.output_projection = nn.Linear(
            projection_width,
            model_dim,
            bias=False,
        )

    def _split_heads(self, x):
        batch_size, sequence_length, _ = x.shape
        return x.view(
            batch_size,
            sequence_length,
            self.heads,
            self.head_dim,
        ).transpose(1, 2)

    def forward(self, hidden):
        q = self.q_up(self.q_norm(self.q_down(hidden)))
        q = self._split_heads(q)

        kv = self.kv_up(self.kv_norm(self.kv_down(hidden)))
        k, v = kv.chunk(2, dim=-1)
        k = self._split_heads(k)
        v = self._split_heads(v)

        attended = F.scaled_dot_product_attention(
            q.float(),
            k.float(),
            v.float(),
            is_causal=True,
        ).to(hidden.dtype)
        attended = attended.transpose(1, 2).contiguous().flatten(2)

        gate = torch.sigmoid(self.output_gate(hidden))
        return self.output_projection(gate * attended)

## 3. SiTU-GLU and Stable LatentMoE

The routed experts are kept at the released count **896** and top-k **16**. Only the vector widths of each expert are reduced. Expert parameters are stored in batched tensors so the faithful expert count remains practical on CPU.

In [ ]:
def situ_gate(x, beta=4.0):
    return beta * torch.tanh(x / beta) * torch.sigmoid(x)


def situ_value(x, beta=25.0):
    return beta * torch.tanh(x / beta)


class DenseSiTUFFN(nn.Module):
    def __init__(self, model_dim=8, intermediate_dim=16):
        super().__init__()
        self.gate = nn.Linear(model_dim, intermediate_dim, bias=False)
        self.value = nn.Linear(model_dim, intermediate_dim, bias=False)
        self.output = nn.Linear(intermediate_dim, model_dim, bias=False)

    def forward(self, hidden):
        gate = situ_gate(self.gate(hidden), beta=4.0)
        value = situ_value(self.value(hidden), beta=25.0)
        return self.output(gate * value)


class BatchedSiTUExperts(nn.Module):
    def __init__(
        self,
        experts,
        input_dim,
        intermediate_dim,
        output_dim,
    ):
        super().__init__()

        self.experts = experts
        scale = 0.02
        self.gate_weight = nn.Parameter(
            scale * torch.randn(
                experts,
                input_dim,
                intermediate_dim,
            )
        )
        self.value_weight = nn.Parameter(
            scale * torch.randn(
                experts,
                input_dim,
                intermediate_dim,
            )
        )
        self.output_weight = nn.Parameter(
            scale * torch.randn(
                experts,
                intermediate_dim,
                output_dim,
            )
        )

    def selected_forward(self, hidden, expert_ids):
        gate_weight = self.gate_weight[expert_ids]
        value_weight = self.value_weight[expert_ids]
        output_weight = self.output_weight[expert_ids]

        gate_pre = torch.einsum(
            "bti,btkif->btkf",
            hidden,
            gate_weight,
        )
        value_pre = torch.einsum(
            "bti,btkif->btkf",
            hidden,
            value_weight,
        )
        intermediate = (
            situ_gate(gate_pre, beta=4.0)
            * situ_value(value_pre, beta=25.0)
        )
        return torch.einsum(
            "btkf,btkfo->btko",
            intermediate,
            output_weight,
        )


class KimiStableLatentMoE(nn.Module):
    def __init__(
        self,
        model_dim=8,
        latent_dim=4,
        routed_intermediate=4,
        shared_intermediate=8,
        experts=896,
        top_k=16,
        shared_experts=2,
    ):
        super().__init__()

        assert experts == 896
        assert top_k == 16
        assert shared_experts == 2

        self.experts = experts
        self.top_k = top_k
        self.shared_expert_count = shared_experts

        self.router = nn.Linear(model_dim, experts, bias=False)
        self.routing_bias = nn.Parameter(
            torch.zeros(experts),
            requires_grad=False,
        )

        self.latent_down = nn.Linear(
            model_dim,
            latent_dim,
            bias=False,
        )
        self.routed_experts = BatchedSiTUExperts(
            experts=experts,
            input_dim=latent_dim,
            intermediate_dim=routed_intermediate,
            output_dim=latent_dim,
        )
        self.latent_norm = KimiRMSNorm(latent_dim)
        self.latent_up = nn.Linear(
            latent_dim,
            model_dim,
            bias=False,
        )

        self.shared_experts = BatchedSiTUExperts(
            experts=shared_experts,
            input_dim=model_dim,
            intermediate_dim=shared_intermediate,
            output_dim=model_dim,
        )

    def forward(self, hidden):
        raw_scores = torch.sigmoid(self.router(hidden))
        selection_scores = raw_scores + self.routing_bias
        expert_ids = selection_scores.topk(
            self.top_k,
            dim=-1,
        ).indices

        selected_scores = raw_scores.gather(-1, expert_ids)
        routing_weights = selected_scores / selected_scores.sum(
            dim=-1,
            keepdim=True,
        ).clamp_min(1e-8)

        latent = self.latent_down(hidden)
        routed = self.routed_experts.selected_forward(
            latent,
            expert_ids,
        )
        routed = (
            routed
            * routing_weights[..., None]
        ).sum(dim=2)
        routed = self.latent_up(self.latent_norm(routed))

        shared_ids = torch.arange(
            self.shared_expert_count,
            device=hidden.device,
        )
        shared_ids = shared_ids.view(1, 1, -1).expand(
            hidden.size(0),
            hidden.size(1),
            -1,
        )
        shared = self.shared_experts.selected_forward(
            hidden,
            shared_ids,
        ).sum(dim=2)

        return routed + shared, raw_scores, expert_ids

## 4. Exact Block Attention Residual ordering

Every attention and FFN sublayer has its own pseudo-query. Raw prefix sums are banked at 12-layer boundaries, and the final model output performs one additional depth-attention read before the final RMSNorm.

In [ ]:
def apply_attention_residual(
    prefix_sum,
    block_residual,
    projection,
    norm,
):
    if block_residual.size(1) == 0:
        return prefix_sum

    values = torch.cat(
        [block_residual, prefix_sum.unsqueeze(1)],
        dim=1,
    )

    values_float = values.float()
    scale = torch.rsqrt(
        values_float.square().mean(dim=-1, keepdim=True)
        + norm.eps
    )
    normalized = values_float * scale

    score_weight = (
        norm.weight.float()
        * projection.weight.squeeze(0).float()
    )
    scores = (
        normalized
        * score_weight[None, None, :]
    ).sum(dim=-1)
    weights = scores.softmax(dim=-1).to(values.dtype)

    return torch.einsum(
        "bk,bkd->bd",
        weights,
        values,
    )


class KimiK3Layer(nn.Module):
    def __init__(
        self,
        layer_index,
        model_dim=8,
        block_size=12,
    ):
        super().__init__()

        self.layer_index = layer_index
        self.block_size = block_size
        layer_number = layer_index + 1

        self.attention_res_norm = KimiRMSNorm(model_dim)
        self.mlp_res_norm = KimiRMSNorm(model_dim)
        self.attention_res_projection = nn.Linear(
            model_dim,
            1,
            bias=False,
        )
        self.mlp_res_projection = nn.Linear(
            model_dim,
            1,
            bias=False,
        )

        self.input_norm = KimiRMSNorm(model_dim)
        self.post_attention_norm = KimiRMSNorm(model_dim)

        if layer_number in K3_FULL_ATTENTION_LAYERS:
            self.attention_kind = "mla"
            self.attention = KimiGatedMLA(model_dim=model_dim)
        else:
            self.attention_kind = "kda"
            self.attention = KimiKDA(model_dim=model_dim)

        if layer_index < 1:
            self.ffn_kind = "dense"
            self.ffn = DenseSiTUFFN(model_dim=model_dim)
        else:
            self.ffn_kind = "moe"
            self.ffn = KimiStableLatentMoE(model_dim=model_dim)

    def forward(self, hidden_states, block_residual):
        batch_size, sequence_length, model_dim = hidden_states.shape
        prefix_sum = hidden_states

        flat_prefix = prefix_sum.reshape(-1, model_dim)
        if block_residual.size(1) > 0:
            hidden_states = apply_attention_residual(
                flat_prefix,
                block_residual,
                self.attention_res_projection,
                self.attention_res_norm,
            ).view(batch_size, sequence_length, model_dim)

        if self.layer_index % self.block_size == 0:
            block_residual = torch.cat(
                [block_residual, flat_prefix.unsqueeze(1)],
                dim=1,
            )
            prefix_sum = None

        attention_input = self.input_norm(hidden_states)
        attention_output = self.attention(attention_input)

        if prefix_sum is None:
            prefix_sum = attention_output
        else:
            prefix_sum = prefix_sum + attention_output

        ffn_input = apply_attention_residual(
            prefix_sum.reshape(-1, model_dim),
            block_residual,
            self.mlp_res_projection,
            self.mlp_res_norm,
        ).view(batch_size, sequence_length, model_dim)
        ffn_input = self.post_attention_norm(ffn_input)

        if self.ffn_kind == "dense":
            ffn_output = self.ffn(ffn_input)
        else:
            ffn_output, _, _ = self.ffn(ffn_input)

        return prefix_sum + ffn_output, block_residual

## 5. Complete 93-layer K3 stack and structural assertions

In [ ]:
class SmallWidthKimiK3(nn.Module):
    def __init__(self, model_dim=8):
        super().__init__()

        self.model_dim = model_dim
        self.block_size = K3_ATTN_RES_BLOCK_SIZE
        self.layers = nn.ModuleList(
            [
                KimiK3Layer(
                    layer_index=layer_index,
                    model_dim=model_dim,
                    block_size=self.block_size,
                )
                for layer_index in range(K3_DEPTH)
            ]
        )
        self.output_res_norm = KimiRMSNorm(model_dim)
        self.output_res_projection = nn.Linear(
            model_dim,
            1,
            bias=False,
        )
        self.final_norm = KimiRMSNorm(model_dim)

    def forward(self, hidden_states):
        batch_size, sequence_length, model_dim = hidden_states.shape
        block_residual = hidden_states.new_zeros(
            batch_size * sequence_length,
            0,
            model_dim,
        )

        for layer in self.layers:
            hidden_states, block_residual = layer(
                hidden_states,
                block_residual,
            )

        hidden_states = apply_attention_residual(
            hidden_states.reshape(-1, model_dim),
            block_residual,
            self.output_res_projection,
            self.output_res_norm,
        ).view(batch_size, sequence_length, model_dim)
        return self.final_norm(hidden_states), block_residual


model = SmallWidthKimiK3().to(device)

assert len(model.layers) == 93
assert model.block_size == 12
assert sum(layer.attention_kind == "kda" for layer in model.layers) == 69
assert sum(layer.attention_kind == "mla" for layer in model.layers) == 24
assert model.layers[0].ffn_kind == "dense"
assert all(layer.ffn_kind == "moe" for layer in model.layers[1:])
assert all(
    layer.ffn.experts == 896
    and layer.ffn.top_k == 16
    and layer.ffn.shared_expert_count == 2
    for layer in model.layers[1:]
)
assert all(
    layer.attention.heads == 96
    for layer in model.layers
)

# Sequence length one is enough to exercise every one of the 93 architectural sites.
hidden = torch.randn(1, 1, 8, device=device)
output, bank = model(hidden)
loss = output.square().mean()
loss.backward()

expected_bank_entries = (93 - 1) // 12 + 1
assert bank.size(1) == expected_bank_entries

print("depth:", len(model.layers))
print("KDA/MLA:",
      sum(layer.attention_kind == "kda" for layer in model.layers),
      sum(layer.attention_kind == "mla" for layer in model.layers))
print("dense/MoE:",
      sum(layer.ffn_kind == "dense" for layer in model.layers),
      sum(layer.ffn_kind == "moe" for layer in model.layers))
print("experts/top-k/shared:", 896, 16, 2)
print("AttnRes bank entries:", bank.size(1))
print("output:", output.shape)

## Structural checklist

The executable assertions verify all architecture-count invariants that must not shrink: 93 layers, the exact 69/24 KDA/MLA schedule, 96 heads, kernel-4 KDA, block-size-12 AttnRes, one dense layer followed by 92 MoE layers, and 896/top-16/2-shared expert routing. Only tensor widths and the sanity-check input length are reduced.

Reference: released `moonshotai/Kimi-K3` configuration and public implementation.